In [ ]:
# loading the dataset and preprosessng data


from imblearn.over_sampling import SMOTE
import pandas as pd
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split

def load_preprocess_with_smote(file_path, n_components=7):
    """
    Load and preprocess the dataset with PCA and apply SMOTE.

    Parameters:
        file_path (str): Path to the dataset CSV file.
        n_components (int): Number of components for PCA.

    Returns:
        X_train_smote, X_test, y_train_smote, y_test, scaler, pca
    """
    # Read the dataset
    df = pd.read_csv(file_path)

    # Remove the 'Name' column if it exists
    if 'Name' in df.columns:
        df = df.drop('Name', axis=1)

    # Convert 'Malware' column to binary (0 for benign, 1 for malware)
    df['Malware'] = df['Malware'].astype(int)

    # Identify non-numeric columns and encode them
    non_numeric_columns = df.select_dtypes(include=['object']).columns
    label_encoder = LabelEncoder()
    for column in non_numeric_columns:
        df[column] = label_encoder.fit_transform(df[column])

    # Separate features and target
    X = df.drop('Malware', axis=1)
    y = df['Malware']

    # Handle missing values by filling with column means
    X = X.fillna(X.mean())

    # Scale the features
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # Apply PCA for dimensionality reduction
    pca = PCA(n_components=n_components)
    X_pca = pca.fit_transform(X_scaled)

    print(f"\nExplained Variance Ratio (PCA): {pca.explained_variance_ratio_}")
    print(f"Total Explained Variance: {sum(pca.explained_variance_ratio_):.4f}")

    # Split the data into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(
        X_pca, y, test_size=0.3, random_state=42, stratify=y
    )

    # Apply SMOTE to balance the training data
    smote = SMOTE(random_state=42)
    X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

    print("\nClass Distribution after SMOTE:")
    print(pd.Series(y_train_smote).value_counts())

    return X_train_smote, X_test, y_train_smote, y_test, scaler, pca

def save_preprocessed_data(X_train, X_test, y_train, y_test):
    """
    Save the preprocessed datasets to CSV files.

    Parameters:
        X_train, X_test, y_train, y_test: Preprocessed datasets.
    """
    pd.DataFrame(X_train).to_csv("X_train.csv", index=False)
    pd.DataFrame(X_test).to_csv("X_test.csv", index=False)
    pd.DataFrame(y_train).to_csv("y_train.csv", index=False)
    pd.DataFrame(y_test).to_csv("y_test.csv", index=False)
    print("\nPreprocessed data saved successfully.")

# Main execution
if __name__ == "__main__":
    file_path = 'dataset_malwares.csv'  # Replace with your dataset file path
    X_train_smote, X_test, y_train_smote, y_test, scaler, pca = load_preprocess_with_smote(file_path)

    # Save the preprocessed data
    save_preprocessed_data(X_train_smote, X_test, y_train_smote, y_test)


Explained Variance Ratio (PCA): [0.1137141  0.06045263 0.05358476 0.04952869 0.04082429 0.03436879
 0.0332004 ]
Total Explained Variance: 0.3857

Class Distribution after SMOTE:
Malware
0    10219
1    10219
Name: count, dtype: int64

Preprocessed data saved successfully.


In [5]:
pip install lightgbm catboost


Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 4.2 MB/s eta 0:00:00m eta 0:00:010:01:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.7/98.7 MB 4.4 MB/s eta 0:00:000m eta 0:00:010:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.8/14.8 MB 4.0 MB/s eta 0:00:00m eta 0:00:010:01:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.1/47.1 KB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 5.6 MB/s eta 0:00:000m eta 0:00:010:01:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 313.1/313.1 KB 208.7 kB/s eta 0:00:000:00:01m eta 0:00:01
  Attempting uninstall: numpy
    Found existing installation: numpy 2.1.3
    Uninstalling numpy-2.1.3:
      Successfully uninstalled numpy-2.1.3
Note: you may need to restart the kernel to use updated packages.


In [ ]:
#stack ensemble model

import numpy as np
import xgboost as xgb
import lightgbm as lgb
import catboost as cb
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.model_selection import KFold
import joblib
import pandas as pd

class AdvancedStackedEnsemble:
    def __init__(self, n_folds=5):
        self.n_folds = n_folds

        # Updated base models with regularization
        self.base_models = {
            'xgb': xgb.XGBClassifier(
                n_estimators=100,
                learning_rate=0.1,
                max_depth=4,
                min_child_weight=5,
                reg_alpha=0.5,
                reg_lambda=1.0,
                random_state=42,
                eval_metric='logloss'
            ),
            'lgb': lgb.LGBMClassifier(
                n_estimators=100,
                learning_rate=0.1,
                max_depth=4,
                reg_alpha=0.5,
                reg_lambda=1.0,
                random_state=42
            ),
            'catboost': cb.CatBoostClassifier(
                iterations=100,
                learning_rate=0.1,
                depth=4,
                random_state=42,
                verbose=False
            ),
            'xgb_dart': xgb.XGBClassifier(
                booster='dart',
                n_estimators=100,
                learning_rate=0.1,
                max_depth=4,
                min_child_weight=5,
                reg_alpha=0.5,
                reg_lambda=1.0,
                random_state=42,
                eval_metric='logloss'
            )
        }

        # Initialize meta-model
        self.meta_model = lgb.LGBMClassifier(
            n_estimators=100,
            learning_rate=0.05,
            max_depth=4,
            random_state=42
        )

        self.base_predictions = {}
        self.feature_importance = {}

    def _get_oof_predictions(self, model, X, y):
        """Get out-of-fold predictions for a single model"""
        oof_predictions = np.zeros(len(X))
        kf = KFold(n_splits=self.n_folds, shuffle=True, random_state=42)

        for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
            X_train_fold, X_val_fold = X[train_idx], X[val_idx]
            y_train_fold, y_val_fold = y[train_idx], y[val_idx]

            # Fit the model
            if isinstance(model, xgb.XGBClassifier):
                try:
                    model.fit(
                        X_train_fold, y_train_fold,
                        eval_set=[(X_val_fold, y_val_fold)],
                        early_stopping_rounds=50,
                        verbose=False
                    )
                except TypeError:
                    # Manual early stopping for older versions of xgboost
                    best_iteration = None
                    best_score = float('inf')
                    for i in range(200):  # Simulate n_estimators
                        model.fit(X_train_fold, y_train_fold)
                        preds = model.predict_proba(X_val_fold)[:, 1]
                        score = roc_auc_score(y_val_fold, preds)
                        if score < best_score:
                            best_score = score
                            best_iteration = i
                        else:
                            break  # Stop training if no improvement
            elif isinstance(model, cb.CatBoostClassifier):
                model.fit(
                    X_train_fold, y_train_fold,
                    eval_set=(X_val_fold, y_val_fold),
                    early_stopping_rounds=50,
                    verbose=False
                )
            else:
                model.fit(X_train_fold, y_train_fold)

            # Get out-of-fold predictions
            oof_predictions[val_idx] = model.predict_proba(X_val_fold)[:, 1]

        return oof_predictions

    def train_base_models(self, X_train, y_train):
        """Train base models and generate meta-features using custom cross-validation"""
        X_train_array = X_train.to_numpy() if isinstance(X_train, pd.DataFrame) else X_train
        meta_features = np.zeros((X_train_array.shape[0], len(self.base_models)))

        for idx, (name, model) in enumerate(self.base_models.items()):
            print(f"\nTraining {name}...")

            # Generate out-of-fold predictions
            meta_features[:, idx] = self._get_oof_predictions(model, X_train_array, y_train)
            self.base_predictions[name] = meta_features[:, idx]

            # Retrain model on full dataset
            if isinstance(model, xgb.XGBClassifier):
                try:
                    model.fit(
                        X_train_array, y_train,
                        eval_set=[(X_train_array, y_train)],
                        early_stopping_rounds=50,
                        verbose=False
                    )
                except TypeError:
                    # Manual early stopping for older versions of xgboost
                    best_iteration = None
                    best_score = float('inf')
                    for i in range(200):  # Simulate n_estimators
                        model.fit(X_train_array, y_train)
                        preds = model.predict_proba(X_train_array)[:, 1]
                        score = roc_auc_score(y_train, preds)
                        if score < best_score:
                            best_score = score
                            best_iteration = i
                        else:
                            break  # Stop training if no improvement
            elif isinstance(model, cb.CatBoostClassifier):
                model.fit(
                    X_train_array, y_train,
                    eval_set=(X_train_array, y_train),
                    early_stopping_rounds=50,
                    verbose=False
                )
            else:
                model.fit(X_train_array, y_train)

            # Store feature importance if available
            if hasattr(model, 'feature_importances_'):
                self.feature_importance[name] = model.feature_importances_

            # Calculate and print metrics
            y_pred = model.predict(X_train_array)
            print(f"{name} training metrics:")
            print(f"Accuracy: {accuracy_score(y_train, y_pred):.4f}")
            print(f"ROC-AUC: {roc_auc_score(y_train, model.predict_proba(X_train_array)[:, 1]):.4f}")

        return meta_features

    def train_meta_model(self, meta_features, y_train):
        """Train the meta-model on base model predictions"""
        print("\nTraining meta-model...")
        self.meta_model.fit(meta_features, y_train)
        print("Meta-model training completed")

    def predict(self, X):
        """Generate predictions using the stacked ensemble"""
        X_array = X.to_numpy() if isinstance(X, pd.DataFrame) else X
        meta_features = np.zeros((X_array.shape[0], len(self.base_models)))

        for idx, (name, model) in enumerate(self.base_models.items()):
            meta_features[:, idx] = model.predict_proba(X_array)[:, 1]

        return self.meta_model.predict(meta_features)

    def predict_proba(self, X):
        """Generate probability predictions with confidence scores"""
        X_array = X.to_numpy() if isinstance(X, pd.DataFrame) else X
        meta_features = np.zeros((X_array.shape[0], len(self.base_models)))
        base_probas = {}

        for name, model in self.base_models.items():
            proba = model.predict_proba(X_array)[:, 1]
            base_probas[name] = proba
            meta_features[:, list(self.base_models.keys()).index(name)] = proba

        final_probas = self.meta_model.predict_proba(meta_features)

        return final_probas, base_probas

    def evaluate(self, X_test, y_test):
        """Evaluate the stacked ensemble model"""
        y_pred = self.predict(X_test)
        y_pred_proba, base_probas = self.predict_proba(X_test)

        metrics = {
            'ensemble_accuracy': accuracy_score(y_test, y_pred),
            'ensemble_precision': precision_score(y_test, y_pred),
            'ensemble_recall': recall_score(y_test, y_pred),
            'ensemble_f1': f1_score(y_test, y_pred),
            'ensemble_roc_auc': roc_auc_score(y_test, y_pred_proba[:, 1])
        }

        for name, probas in base_probas.items():
            pred = (probas > 0.5).astype(int)
            metrics[f'{name}_accuracy'] = accuracy_score(y_test, pred)
            metrics[f'{name}_roc_auc'] = roc_auc_score(y_test, probas)

        return metrics

def train_and_evaluate_ensemble(X_train, X_test, y_train, y_test):
    """Train and evaluate the advanced stacked ensemble"""
    ensemble = AdvancedStackedEnsemble()

    # Train base models and generate meta-features
    meta_features = ensemble.train_base_models(X_train, y_train)

    # Train meta-model
    ensemble.train_meta_model(meta_features, y_train)

    # Evaluate model
    print("\nEvaluating models...")
    metrics = ensemble.evaluate(X_test, y_test)

    print("\nModel Performance:")
    for metric, value in metrics.items():
        print(f"{metric}: {value:.4f}")

    return ensemble

def save_model(model, filename='advanced_stacked_ensemble.joblib'):
    """Save the trained model"""
    joblib.dump(model, filename)
    print(f"\nModel saved as {filename}")

# Main execution
if __name__ == "__main__":
    # Load your preprocessed data
    X_train = pd.read_csv("X_train.csv")
    X_test = pd.read_csv("X_test.csv")
    y_train = pd.read_csv("y_train.csv").iloc[:, 0]
    y_test = pd.read_csv("y_test.csv").iloc[:, 0]

    # Train and evaluate the ensemble
    ensemble = train_and_evaluate_ensemble(X_train, X_test, y_train, y_test)

    # Save the model
    save_model(ensemble)


Training xgb...
xgb training metrics:
Accuracy: 0.9746
ROC-AUC: 0.9955

Training lgb...
[LightGBM] [Info] Number of positive: 8229, number of negative: 8121
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001096 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1785
[LightGBM] [Info] Number of data points in the train set: 16350, number of used features: 7
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.503303 -> initscore=0.013211
[LightGBM] [Info] Start training from score 0.013211
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with pos

/home/sanjeev/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sanjeev/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 8183, number of negative: 8167
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001072 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1785
[LightGBM] [Info] Number of data points in the train set: 16350, number of used features: 7
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500489 -> initscore=0.001957
[LightGBM] [Info] Start training from score 0.001957
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, be

/home/sanjeev/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sanjeev/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 8186, number of negative: 8165
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001066 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1785
[LightGBM] [Info] Number of data points in the train set: 16351, number of used features: 7
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500642 -> initscore=0.002569
[LightGBM] [Info] Start training from score 0.002569
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, be

...

[LightGBM] [Info] Number of positive: 10219, number of negative: 10219
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000953 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1785
[LightGBM] [Info] Number of data points in the train set: 20438, number of used features: 7
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further split

/home/sanjeev/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


catboost training metrics:
Accuracy: 0.9657
ROC-AUC: 0.9892

Training xgb_dart...
xgb_dart training metrics:
Accuracy: 0.9746
ROC-AUC: 0.9955

Training meta-model...
[LightGBM] [Info] Number of positive: 10219, number of negative: 10219
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.044490 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1020
[LightGBM] [Info] Number of data points in the train set: 20438, number of used features: 4
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] 

/home/sanjeev/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sanjeev/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sanjeev/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sanjeev/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



Model Performance:
ensemble_accuracy: 0.9743
ensemble_precision: 0.9822
ensemble_recall: 0.9833
ensemble_f1: 0.9828
ensemble_roc_auc: 0.9889
xgb_accuracy: 0.9711
xgb_roc_auc: 0.9871
lgb_accuracy: 0.9718
lgb_roc_auc: 0.9885
catboost_accuracy: 0.9641
catboost_roc_auc: 0.9830
xgb_dart_accuracy: 0.9711
xgb_dart_roc_auc: 0.9871

Model saved as advanced_stacked_ensemble.joblib


created advanced stakve embbsems and bagging voting ensemble


In [2]:
pip install --upgrade xgboost


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [ ]:
pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.7/98.7 MB 7.7 MB/s eta 0:00:00


In [ ]:
# accuracy presision metrics

import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)
from sklearn.model_selection import train_test_split
import joblib
import os

# Load the saved ensemble model
def load_model(filename='advanced_stacked_ensemble.joblib'):
    """Load the trained model"""
    print(f"\nLoading model from {filename}...")
    return joblib.load(filename)

# Evaluate the loaded model on test data
def evaluate_saved_model(model, X_test, y_test):
    """Evaluate the loaded stacked ensemble model on test data"""
    # Generate predictions
    y_pred = model.predict(X_test)
    y_pred_proba, base_probas = model.predict_proba(X_test)

    # Calculate metrics
    metrics = {
        'ensemble_accuracy': accuracy_score(y_test, y_pred),
        'ensemble_precision': precision_score(y_test, y_pred),
        'ensemble_recall': recall_score(y_test, y_pred),
        'ensemble_f1': f1_score(y_test, y_pred),
        'ensemble_roc_auc': roc_auc_score(y_test, y_pred_proba[:, 1])
    }

    for name, probas in base_probas.items():
        pred = (probas > 0.5).astype(int)
        metrics[f'{name}_accuracy'] = accuracy_score(y_test, pred)
        metrics[f'{name}_roc_auc'] = roc_auc_score(y_test, probas)

    # Confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    print("\nConfusion Matrix:\n", cm)

    # Classification report
    print("\nClassification Report:\n", classification_report(y_test, y_pred))

    return metrics

# Main execution
if __name__ == "__main__":
    # Debugging: Print current working directory and list files
    print("Current Working Directory:", os.getcwd())
    print("Files in Directory:", os.listdir())

    # Load your preprocessed data
    try:
        # Use the correct file names based on your directory listing
        X_train = pd.read_csv("X_train.csv")  # Training features
        X_test = pd.read_csv("X_test.csv")   # Test features
        y_train = pd.read_csv("y_train.csv").iloc[:, 0]  # Training labels
        y_test = pd.read_csv("y_test.csv").iloc[:, 0]    # Test labels
    except FileNotFoundError as e:
        print("Error:", e)
        print("Please ensure that 'X_train.csv', 'X_test.csv', 'y_train.csv', and 'y_test.csv' are in the correct directory.")
        exit(1)

    # Print class distribution to check for imbalance
    print("Class distribution in y_train:\n", y_train.value_counts())
    print("Class distribution in y_test:\n", y_test.value_counts())

    # Load the saved model
    ensemble = load_model('advanced_stacked_ensemble.joblib')

    # Evaluate the model on test data
    print("\nEvaluating the saved model on test data...")
    metrics = evaluate_saved_model(ensemble, X_test, y_test)

    # Print performance metrics
    print("\nModel Performance on Test Data:")
    for metric, value in metrics.items():
        print(f"{metric}: {value:.4f}")

Current Working Directory: /content
Files in Directory: ['.config', 'X_test.csv', 'y_train.csv', 'dataset_malwares.csv', 'advanced_stacked_ensemble.joblib', 'catboost_info', 'y_test.csv', 'X_train.csv', 'sample_data']
Class distribution in y_train:
 Malware
0    10219
1    10219
Name: count, dtype: int64
Class distribution in y_test:
 Malware
1    4380
0    1504
Name: count, dtype: int64

Loading model from advanced_stacked_ensemble.joblib...

Evaluating the saved model on test data...


/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(



Confusion Matrix:
 [[1426   78]
 [  73 4307]]

Classification Report:
               precision    recall  f1-score   support

           0       0.95      0.95      0.95      1504
           1       0.98      0.98      0.98      4380

    accuracy                           0.97      5884
   macro avg       0.97      0.97      0.97      5884
weighted avg       0.97      0.97      0.97      5884


Model Performance on Test Data:
ensemble_accuracy: 0.9743
ensemble_precision: 0.9822
ensemble_recall: 0.9833
ensemble_f1: 0.9828
ensemble_roc_auc: 0.9889
xgb_accuracy: 0.9711
xgb_roc_auc: 0.9871
lgb_accuracy: 0.9718
lgb_roc_auc: 0.9885
catboost_accuracy: 0.9641
catboost_roc_auc: 0.9830
xgb_dart_accuracy: 0.9711
xgb_dart_roc_auc: 0.9871


/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


In [ ]:
#voting and prediction

import numpy as np
import pandas as pd
from sklearn.ensemble import BaggingClassifier, VotingClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score
import joblib

# Load preprocessed data
X_train = pd.read_csv("X_train.csv")
X_test = pd.read_csv("X_test.csv")
y_train = pd.read_csv("y_train.csv").iloc[:, 0]
y_test = pd.read_csv("y_test.csv").iloc[:, 0]

# Define base models for Bagging
base_models = {
    "DecisionTree": DecisionTreeClassifier(max_depth=6, random_state=42),
    "RandomForest": RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42),
    "XGBoost": XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=6, random_state=42),
    "LightGBM": LGBMClassifier(n_estimators=100, learning_rate=0.1, max_depth=6, random_state=42),
}

# Train Bagging classifiers
bagging_models = {
    name: BaggingClassifier(estimator=model, n_estimators=10, random_state=42)
    for name, model in base_models.items()
}

# Train all models
for name, model in bagging_models.items():
    print(f"Training {name} with Bagging...")
    model.fit(X_train, y_train)

# Voting Classifier with Hard Voting
voting_clf = VotingClassifier(
    estimators=[(name, model) for name, model in bagging_models.items()],
    voting="hard"  # Use "soft" for probability-based voting
)

# Train Voting Classifier
print("\nTraining Voting Classifier...")
voting_clf.fit(X_train, y_train)

# Make predictions
y_pred = voting_clf.predict(X_test)

# Convert numeric predictions to labels
final_predictions = ["Malware" if pred == 1 else "Benign" for pred in y_pred]

# Evaluate the model
metrics = {
    "accuracy": accuracy_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred),
    "recall": recall_score(y_test, y_pred),
    "f1_score": f1_score(y_test, y_pred),
    "roc_auc": roc_auc_score(y_test, y_pred),
}

# Print performance metrics
print("\nModel Performance:")
for metric, value in metrics.items():
    print(f"{metric}: {value:.4f}")

# Save model
joblib.dump(voting_clf, "bagging_voting_ensemble.joblib")
print("\nModel saved as 'bagging_voting_ensemble.joblib'")

# Save predictions to a file
output_df = pd.DataFrame({"File": range(1, len(final_predictions) + 1), "Prediction": final_predictions})
output_df.to_csv("final_predictions.csv", index=False)

# Print final predictions
print("\nFinal Predictions:")
for index, label in enumerate(final_predictions):
    print(f"File {index+1}: {label}")


Training DecisionTree with Bagging...
Training RandomForest with Bagging...
Training XGBoost with Bagging...
Training LightGBM with Bagging...
[LightGBM] [Info] Number of positive: 10219, number of negative: 10219
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000844 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1785
[LightGBM] [Info] Number of data points in the train set: 20438, number of used features: 7
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.495939 -> initscore=-0.016245
[LightGBM] [Info] Start training from score -0.016245
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Info] Number of positive: 10219, number of negative: 10219
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000911 seconds.
You can set `force_col_wise=true` to re

/home/sanjeev/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sanjeev/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sanjeev/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sanjeev/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sanjeev/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



Model Performance:
accuracy: 0.9652
precision: 0.9849
recall: 0.9680
f1_score: 0.9764
roc_auc: 0.9624

Model saved as 'bagging_voting_ensemble.joblib'

Final Predictions:
File 1: Malware
File 2: Malware
File 3: Malware
File 4: Malware
File 5: Malware
File 6: Benign
File 7: Benign
File 8: Malware
File 9: Malware
File 10: Benign
File 11: Benign
File 12: Benign
File 13: Benign
File 14: Benign
File 15: Malware
File 16: Malware
File 17: Malware
File 18: Malware
File 19: Malware
File 20: Malware
File 21: Malware
File 22: Benign
File 23: Malware
File 24: Benign
File 25: Malware
File 26: Malware
File 27: Benign
File 28: Benign
File 29: Malware
File 30: Malware
File 31: Benign
File 32: Malware
File 33: Malware
File 34: Benign
File 35: Malware
File 36: Malware
File 37: Malware
File 38: Malware
File 39: Malware
File 40: Malware
File 41: Malware
File 42: Malware
File 43: Benign
File 44: Benign
File 45: Malware
File 46: Malware
File 47: Benign
File 48: Benign
File 49: Malware
File 50: Malware
File